In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"


## Reasoning(OpenMath) and Non-reasoning(Finetome) reasoning

In [3]:
# import torch
# import gc
# import re
# from datasets import load_dataset
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from sklearn.metrics import accuracy_score, f1_score
# from rouge_score import rouge_scorer
# from tqdm import tqdm
# from unsloth import FastLanguageModel
# # ─────────────────────────────────────────────
# # OPENMATHREASONING EVALUATION
# # ─────────────────────────────────────────────

# def extract_math_answer(text: str) -> str:
#     """Extract final answer — tries \\boxed{} first, then last number."""
#     # Try \boxed{...}
#     boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
#     if boxed:
#         return boxed[-1].strip()
#     # Try #### answer format (common in math datasets)
#     hash_match = re.findall(r"####\s*([^\n]+)", text)
#     if hash_match:
#         return hash_match[-1].strip()
#     # Fall back to last number in text
#     numbers = re.findall(r"-?\d+\.?\d*", text)
#     if numbers:
#         return numbers[-1].strip()
#     return text.strip()


# def format_math_prompt(problem: str) -> str:
#     return (
#         f"Solve the following math problem. "
#         f"Show your reasoning and put your final answer in \\boxed{{}}.\n\n"
#         f"Problem: {problem}\n\n"
#         f"Solution:"
#     )


# def evaluate_math(
#     model,
#     tokenizer,
#     model_name: str = "model",
#     num_samples: int = 200,
#     batch_size: int = 4,
#     max_new_tokens: int = 256,
#     device: str = "cuda",
# ) -> dict:
#     """Evaluate on OpenMathReasoning-mini using exact match on final answer."""
#     print(f"\n{'─'*60}")
#     print(f"[Math] Evaluating: {model_name}")
#     print(f"{'─'*60}")

#     model.eval()

#     dataset = load_dataset("unsloth/OpenMathReasoning-mini", split="cot")
#     dataset = dataset.select(range(min(num_samples, len(dataset))))

#     # Check column names
#     print(f"Columns: {dataset.column_names}")

#     preds  = []
#     labels = []

#     for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [math]"):
#         batch = dataset[i : i + batch_size]

#         # OpenMathReasoning columns: problem, solution, answer
#         problems       = batch["problem"]
#         true_answers   = [extract_math_answer(a) for a in batch["expected_answer"]]

#         prompts = [format_math_prompt(p) for p in problems]

#         inputs = tokenizer(
#             prompts,
#             return_tensors = "pt",
#             padding        = True,
#             truncation     = True,
#             max_length     = 512,
#         ).to(device)

#         with torch.no_grad():
#             outputs = model.generate(
#                 **inputs,
#                 max_new_tokens     = max_new_tokens,
#                 do_sample          = False,
#                 pad_token_id       = tokenizer.eos_token_id,
#                 eos_token_id       = tokenizer.eos_token_id,
#                 repetition_penalty = 1.3,
#             )

#         for j, output in enumerate(outputs):
#             input_len   = inputs["input_ids"].shape[1]
#             generated   = tokenizer.decode(output[input_len:], skip_special_tokens=True)
#             pred_answer = extract_math_answer(generated)
#             preds.append(pred_answer)
#             labels.append(true_answers[j])

#     # Exact match
#     exact_matches = [p.strip() == l.strip() for p, l in zip(preds, labels)]
#     exact_match   = round(sum(exact_matches) / len(exact_matches), 4)

#     result = {
#         "repo_id"      : model_name,
#         "exact_match"  : exact_match,
#         "num_samples"  : num_samples,
#         "sample_preds" : list(zip(labels[:5], preds[:5])),  # first 5 for inspection
#     }

#     print(f"  Exact Match: {exact_match:.4f}")
#     print(f"  Sample predictions (true → pred):")
#     for true, pred in result["sample_preds"]:
#         print(f"    {true:<20} → {pred}")

#     return result


# # ─────────────────────────────────────────────
# # FINETOME EVALUATION (ROUGE)
# # ─────────────────────────────────────────────

# def format_finetome_prompt(conversation: list) -> tuple[str, str]:
#     """
#     Extract user prompt and reference response from conversation turns.
#     FineTome-100k has a 'conversations' field with role/value pairs.
#     Returns (prompt, reference_response).
#     """
#     prompt    = ""
#     reference = ""

#     for turn in conversation:
#         role  = turn.get("from", turn.get("role", ""))
#         value = turn.get("value", turn.get("content", ""))
#         if role in ("human", "user") and not prompt:
#             prompt = f"User: {value}\n\nAssistant:"
#         elif role in ("gpt", "assistant") and not reference:
#             reference = value

#     return prompt, reference


# def evaluate_finetome(
#     model,
#     tokenizer,
#     model_name: str = "model",
#     num_samples: int = 200,
#     batch_size: int = 4,
#     max_new_tokens: int = 256,
#     device: str = "cuda",
# ) -> dict:
#     """Evaluate on FineTome-100k using ROUGE-1, ROUGE-2, ROUGE-L."""
#     print(f"\n{'─'*60}")
#     print(f"[FineTome] Evaluating: {model_name}")
#     print(f"{'─'*60}")

#     model.eval()

#     dataset = load_dataset("mlabonne/FineTome-100k", split="train")
#     dataset = dataset.shuffle(seed=42).select(range(min(num_samples, len(dataset))))

#     print(f"Columns: {dataset.column_names}")

#     scorer    = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
#     all_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

#     for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [finetome]"):
#         batch      = dataset[i : i + batch_size]
#         prompts    = []
#         references = []

#         for conv in batch["conversations"]:
#             prompt, reference = format_finetome_prompt(conv)
#             prompts.append(prompt)
#             references.append(reference)

#         # Skip if no valid prompts extracted
#         if not any(prompts):
#             continue

#         inputs = tokenizer(
#             prompts,
#             return_tensors = "pt",
#             padding        = True,
#             truncation     = True,
#             max_length     = 512,
#         ).to(device)

#         with torch.no_grad():
#             outputs = model.generate(
#                 **inputs,
#                 max_new_tokens     = max_new_tokens,
#                 do_sample          = False,
#                 pad_token_id       = tokenizer.eos_token_id,
#                 eos_token_id       = tokenizer.eos_token_id,
#                 repetition_penalty = 1.3,
#             )

#         for j, output in enumerate(outputs):
#             input_len = inputs["input_ids"].shape[1]
#             generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)

#             if references[j]:
#                 scores = scorer.score(references[j], generated)
#                 all_scores["rouge1"].append(scores["rouge1"].fmeasure)
#                 all_scores["rouge2"].append(scores["rouge2"].fmeasure)
#                 all_scores["rougeL"].append(scores["rougeL"].fmeasure)

#     result = {
#         "repo_id"    : model_name,
#         "rouge1"     : round(sum(all_scores["rouge1"]) / len(all_scores["rouge1"]), 4),
#         "rouge2"     : round(sum(all_scores["rouge2"]) / len(all_scores["rouge2"]), 4),
#         "rougeL"     : round(sum(all_scores["rougeL"]) / len(all_scores["rougeL"]), 4),
#         "num_samples": num_samples,
#     }

#     print(f"  ROUGE-1: {result['rouge1']:.4f}")
#     print(f"  ROUGE-2: {result['rouge2']:.4f}")
#     print(f"  ROUGE-L: {result['rougeL']:.4f}")

#     return result


# # ─────────────────────────────────────────────
# # EVALUATE ALL MERGED MODELS
# # ─────────────────────────────────────────────

# def evaluate_all_qwen_models(
#     repos: list[str],
#     num_samples: int = 200,
#     batch_size: int = 4,
#     device: str = "cuda",
# ) -> dict:
#     all_results = {}

#     for repo in repos:
#         print(f"\n{'═'*60}")
#         print(f"Model: {repo}")
#         print(f"{'═'*60}")

#         # ← use Unsloth instead of AutoModelForCausalLM
#         model, tokenizer = FastLanguageModel.from_pretrained(
#             model_name     = repo,
#             max_seq_length = 1024,
#             load_in_4bit   = True,
#             dtype          = torch.float16,
#         )
#         FastLanguageModel.for_inference(model)  # ← required for fast generation

#         math_result     = evaluate_math(model, tokenizer, model_name=repo,
#                                         num_samples=num_samples, batch_size=batch_size)
#         finetome_result = evaluate_finetome(model, tokenizer, model_name=repo,
#                                             num_samples=num_samples, batch_size=batch_size)

#         all_results[repo] = {
#             "math"    : math_result,
#             "finetome": finetome_result,
#         }

#         del model, tokenizer
#         gc.collect()
#         torch.cuda.empty_cache()

#     # ── Summary table ──
#     print(f"\n{'═'*70}")
#     print(f"{'MODEL':<35} {'EXACT':>7} {'R1':>7} {'R2':>7} {'RL':>7}")
#     print(f"{'─'*70}")
#     for repo, r in all_results.items():
#         name = repo.split("/")[-1]
#         print(
#             f"{name:<35} "
#             f"{r['math']['exact_match']:>7.4f} "
#             f"{r['finetome']['rouge1']:>7.4f} "
#             f"{r['finetome']['rouge2']:>7.4f} "
#             f"{r['finetome']['rougeL']:>7.4f}"
#         )
#     print(f"{'═'*70}")

#     return all_results
# # ── Usage ──

# repos = [
#     "Srishtik/Qwen3-0.6B-linear-3-adapters-merged",
#     "Srishtik/Qwen3-0.6B-svd-3-adapters-merged",
#     "Srishtik/Qwen3-0.6B-ties-3-adapters-merged",
#     "Srishtik/Qwen3-0.6B-dare-3-adapters-merged",
#     "Srishtik/Qwen3-0.6B-slerp-3-adapters-merged",
# ]

# all_results = evaluate_all_qwen_models(
#     repos       = repos,
#     num_samples = 200,
#     batch_size  = 4,
# )

## AG News evaluation

In [4]:
import warnings
warnings.filterwarnings("ignore")

In [5]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [6]:


import torch
import gc
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

# ── AG News label map ──
LABEL_MAP = {
    "1": "World",
    "2": "Sports", 
    "3": "Business",
    "4": "Sci/Tech",
    "World": "World",
    "Sports": "Sports",
    "Business": "Business",
    "Sci/Tech": "Sci/Tech",
}

INT_TO_LABEL = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

def format_prompt(title: str, description: str) -> str:
    return (
        f"Classify the following news article into one of these categories: "
        f"World, Sports, Business, Sci/Tech.\n\n"
        f"Title: {title}\n"
        f"Description: {description}\n\n"
        f"Category:"
    )

def extract_label(generated_text: str) -> str:
    """Extract the predicted label from generated text."""
    text = generated_text.strip()
    for label in ["Sci/Tech", "Business", "Sports", "World"]:  # longer first to avoid partial match
        if label.lower() in text.lower():
            return label
    return "World"  # fallback


def evaluate_on_agnews(
    repo_id: str,
    tokenizer,
    num_samples: int = 500,
    batch_size: int = 8,
    max_new_tokens: int = 10,
    device: str = "cuda",
) -> dict:
    """
    Evaluate a merged model on AG News test set.

    Args:
        repo_id       : HuggingFace repo to evaluate
        tokenizer     : shared tokenizer
        num_samples   : number of test samples (full test = 7600)
        batch_size    : inference batch size
        max_new_tokens: how many tokens to generate for label
        device        : cuda or cpu

    Returns:
        dict with accuracy, macro_f1, per_class_f1, repo_id
    """
    print(f"\n{'─'*60}")
    print(f"Evaluating: {repo_id}")
    print(f"{'─'*60}")

    # ── Load model ──
    model = AutoModelForCausalLM.from_pretrained(
        repo_id,
        torch_dtype=torch.float16,
        device_map=device,
    )
    model.eval()

    # ── Load dataset ──
    dataset = load_dataset("ag_news", split="test")
    dataset = dataset.select(range(num_samples))

    preds  = []
    labels = []

    # ── Inference in batches ──
    for i in tqdm(range(0, len(dataset), batch_size), desc=repo_id.split("/")[-1]):
        batch = dataset[i : i + batch_size]

        prompts = [
            format_prompt(title, desc)
            for title, desc in zip(batch["text"], batch["text"])  # ag_news has no separate title field
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,        # greedy for reproducibility
                pad_token_id=tokenizer.eos_token_id,
            )

        # Decode only the generated part (strip the prompt)
        for j, output in enumerate(outputs):
            input_len  = inputs["input_ids"].shape[1]
            generated  = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            pred_label = extract_label(generated)
            true_label = INT_TO_LABEL[batch["label"][j]]

            preds.append(pred_label)
            labels.append(true_label)

    # ── Compute metrics ──
    label_names   = ["World", "Sports", "Business", "Sci/Tech"]
    accuracy      = accuracy_score(labels, preds)
    macro_f1      = f1_score(labels, preds, average="macro",    labels=label_names, zero_division=0)
    per_class_f1  = f1_score(labels, preds, average=None,       labels=label_names, zero_division=0)

    result = {
        "repo_id"      : repo_id,
        "accuracy"     : round(accuracy, 4),
        "macro_f1"     : round(macro_f1, 4),
        "per_class_f1" : {
            label: round(float(score), 4)
            for label, score in zip(label_names, per_class_f1)
        },
        "num_samples"  : num_samples,
    }

    print(f"  Accuracy  : {result['accuracy']:.4f}")
    print(f"  Macro F1  : {result['macro_f1']:.4f}")
    for label, score in result["per_class_f1"].items():
        print(f"  F1 {label:<10}: {score:.4f}")

    # ── Free memory ──
    del model
    gc.collect()
    torch.cuda.empty_cache()

    return result


def evaluate_all_models(
    repos: list[str],
    tokenizer,
    num_samples: int = 500,
    batch_size: int = 8,
) -> dict:
    """
    Evaluate all merged models sequentially, freeing memory between each.

    Args:
        repos       : list of HuggingFace repo ids
        tokenizer   : shared tokenizer
        num_samples : test samples per model
        batch_size  : inference batch size

    Returns:
        dict mapping repo_id → metrics
    """
    all_results = {}

    for repo in repos:
        result = evaluate_on_agnews(
            repo_id     = repo,
            tokenizer   = tokenizer,
            num_samples = num_samples,
            batch_size  = batch_size,
        )
        all_results[repo] = result

    # ── Summary table ──
    print(f"\n{'═'*60}")
    print(f"{'MODEL':<35} {'ACC':>6} {'F1':>6}")
    print(f"{'─'*60}")
    for repo, r in all_results.items():
        name = repo.split("/")[-1]
        print(f"{name:<35} {r['accuracy']:>6.4f} {r['macro_f1']:>6.4f}")
    print(f"{'═'*60}")

    return all_results


# ── Usage ──

tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3-0.6B")

repos = [
    "Srishtik/Qwen3-0.6B-linear-3-adapters-merged",
    "Srishtik/Qwen3-0.6B-svd-3-adapters-merged",
    "Srishtik/Qwen3-0.6B-ties-3-adapters-merged",
    "Srishtik/Qwen3-0.6B-dare-3-adapters-merged",
    "Srishtik/Qwen3-0.6B-slerp-3-adapters-merged",
]

all_results = evaluate_all_models(
    repos       = repos,
    tokenizer   = tokenizer,
    num_samples = 500,    # increase to 7600 for full test set
    batch_size  = 8,
)



config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

Qwen3-0.6B-slerp-3-adapters-merged: 100%|██████████| 63/63 [00:49<00:00,  1.27it/s]


  Accuracy  : 0.5700
  Macro F1  : 0.5277
  F1 World     : 0.2917
  F1 Sports    : 0.8643
  F1 Business  : 0.4302
  F1 Sci/Tech  : 0.5248

════════════════════════════════════════════════════════════
MODEL                                  ACC     F1
────────────────────────────────────────────────────────────
Qwen3-0.6B-linear-3-adapters-merged 0.4400 0.4123
Qwen3-0.6B-svd-3-adapters-merged    0.4260 0.3984
Qwen3-0.6B-ties-3-adapters-merged   0.4080 0.3284
Qwen3-0.6B-dare-3-adapters-merged   0.3160 0.2536
Qwen3-0.6B-slerp-3-adapters-merged  0.5700 0.5277
════════════════════════════════════════════════════════════


In [7]:
from peft import set_peft_model_state_dict
from huggingface_hub import hf_hub_download
from unsloth import FastLanguageModel
def load_adapter(huggingface_repo):
    model,tokenizer=FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen3-0.6B",
        max_seq_length=1024,
        load_in_4bit=True,
    )
    model=FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                       "up_proj","down_proj","gate_proj"],
        lora_alpha=32,
        lora_dropout=0,
        use_rslora=False,
    )
    try:
        model_weights=hf_hub_download(
         repo_id=f"Srishtik/{huggingface_repo}",
         filename="adapter_model.safetensors"
        )
    except:
        model_weights=hf_hub_download(
         repo_id=f"Srishtik/{huggingface_repo}",
         filename="adapter_model.bin"
        )
    from safetensors.torch import load_file
    model_weights=load_file(model_weights)
    set_peft_model_state_dict(model,model_weights)
    return model,tokenizer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [8]:
full_model,full_tokenizer=load_adapter("3-adapter-merge-qwen-3-0.6B-ag-news-10k")

==((====))==  Unsloth 2026.6.7: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/576M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-0.6b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


adapter_model.safetensors:   0%|          | 0.00/40.4M [00:00<?, ?B/s]

In [14]:
model,tokenizer=FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen3-0.6B",
        max_seq_length=1024,
        load_in_4bit=True,
    )
model=FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                       "up_proj","down_proj","gate_proj"],
        lora_alpha=32,
        lora_dropout=0,
        use_rslora=False,
    )

==((====))==  Unsloth 2026.6.7: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

unsloth/qwen3-0.6b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [11]:


def format_prompt(text: str) -> str:
    return (
        f"Classify the following news article into one of these categories: "
        f"World, Sports, Business, Sci/Tech.\n\n"
        f"Article: {text}\n\n"
        f"Category:"
    )



In [12]:
def evaluate_initialized_model(
    model,
    tokenizer,
    model_name: str = "full_model",
    num_samples: int = 500,
    batch_size: int = 8,
    max_new_tokens: int = 10,
    device: str = "cuda",
) -> dict:
    """
    Evaluate an already-loaded model on AG News.
    Does NOT load or delete the model — caller manages memory.
    """
    print(f"\n{'─'*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'─'*60}")

    model.eval()

    dataset = load_dataset("ag_news", split="test")
    dataset = dataset.select(range(num_samples))

    preds  = []
    labels = []

    for i in tqdm(range(0, len(dataset), batch_size), desc=model_name):
        batch = dataset[i : i + batch_size]

        prompts = [format_prompt(text) for text in batch["text"]]

        inputs = tokenizer(
            prompts,
            return_tensors = "pt",
            padding        = True,
            truncation     = True,
            max_length     = 512,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens = max_new_tokens,
                do_sample      = False,
                pad_token_id   = tokenizer.eos_token_id,
            )

        for j, output in enumerate(outputs):
            input_len  = inputs["input_ids"].shape[1]
            generated  = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            pred_label = extract_label(generated)
            true_label = INT_TO_LABEL[batch["label"][j]]

            preds.append(pred_label)
            labels.append(true_label)

    label_names  = ["World", "Sports", "Business", "Sci/Tech"]
    accuracy     = accuracy_score(labels, preds)
    macro_f1     = f1_score(labels, preds, average="macro", labels=label_names, zero_division=0)
    per_class_f1 = f1_score(labels, preds, average=None,    labels=label_names, zero_division=0)

    result = {
        "repo_id"      : model_name,
        "accuracy"     : round(accuracy, 4),
        "macro_f1"     : round(macro_f1, 4),
        "per_class_f1" : {
            label: round(float(score), 4)
            for label, score in zip(label_names, per_class_f1)
        },
        "num_samples"  : num_samples,
    }

    print(f"  Accuracy  : {result['accuracy']:.4f}")
    print(f"  Macro F1  : {result['macro_f1']:.4f}")
    for label, score in result["per_class_f1"].items():
        print(f"  F1 {label:<10}: {score:.4f}")

    return result



In [13]:


result = evaluate_initialized_model(
    model      = full_model,
    tokenizer  = full_tokenizer,
    num_samples = 500,
    batch_size  = 8,
)




────────────────────────────────────────────────────────────
Evaluating: full_model
────────────────────────────────────────────────────────────


full_model: 100%|██████████| 63/63 [01:34<00:00,  1.50s/it]

  Accuracy  : 0.4740
  Macro F1  : 0.4566
  F1 World     : 0.2717
  F1 Sports    : 0.6361
  F1 Business  : 0.3643
  F1 Sci/Tech  : 0.5541


In [17]:


result = evaluate_initialized_model(
    model      = model,
    tokenizer  = tokenizer,
    model_name = "base_model",
    num_samples = 500,
    batch_size  = 8,
)




────────────────────────────────────────────────────────────
Evaluating: base_model
────────────────────────────────────────────────────────────


base_model: 100%|██████████| 63/63 [01:29<00:00,  1.43s/it]

  Accuracy  : 0.2120
  Macro F1  : 0.0911
  F1 World     : 0.0000
  F1 Sports    : 0.0000
  F1 Business  : 0.3488
  F1 Sci/Tech  : 0.0156
